This notebook contains code for prompting LLMs to translate NL to structured NL or TL

Currently, the notebook is set up to generate FRETish, and you can all cells to test it
You can configure to select the benchmark, and target structured NL (fretish or PSP)

In the cell below, you must provide your GEMINI or OPENAI API KEY

To convenientialy run all cells: Select Run -> Run all cells option from the top of this notebook to view the plots

In [14]:
import os
os.environ["STRUCTNL_MODE"] = "fretish" #or "PSP"
#No config needed, since we're using Ollama
#os.environ["GEMINI_API_KEY"] ="TODO"
#os.environ["OPENAI_API_KEY"] ="TODO"
# Optionally override Ollama URL (default: http://localhost:11434/v1)
#os.environ["OLLAMA_BASE_URL"] = "http://localhost:11434/v1"

In [15]:
import pandas as pd
import json
import itertools
os.environ["DATA_HOME_DIR"] = "./metadata"
#os.environ["GOOGLE_APPLICATION_CREDENTIALS"] =""

In [16]:
import data_loader
import llm_prompt
import nl2structnl

In [17]:
if os.getenv("STRUCTNL_MODE") == "fretish":
    data_home_dir = "./benchmarks/fret_specs/"
else:
    data_home_dir = "./benchmarks/psp_specs/"


In [18]:
#select the list of benchmarks

#dataset_list = ["Ventilator","FSM-AP","FSM-S","REG","RobotExplain","deepstl-test"]
#dataset_list = ["Thales"]
dataset_list = ["FSM-S"]

In [19]:
#select model and number of candidates to generate per requirement

num_trial = 1
model_list = ["qwen2.5:32b"]
#model_list = ["gemini-2.5-flash"]
#model_list = ["gpt-4.1"]

# Set True to always generate atomic propositions via Ollama (ignores Variables.xlsx).
# Set False to only generate when Variables.xlsx is not present.
ALWAYS_GENERATE_VARIABLES = True

In [20]:
#specify directory to save results

save_dir = "./ollama_outputs"

In [21]:
#specify prompting approach, currently configured to use ARTEMIS "nl2structnl"

all_methods = ["nl2structnl","nl2ltl","nl2spec","NL2TL","NL2TL-FT","deepstl","nl2ltltemplate","synthtl"]
cur_methods = ["nl2structnl"]
cur_methods

['nl2structnl']

In [ ]:
import time
def run_single_task(cur_dataset_name,model,row_idx, cur_method):
    cur_df_file = data_home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
    df = pd.read_excel(cur_df_file, engine='openpyxl')
    cur_exp_name = f"{save_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
    #if row_idx < len(df) and not os.path.exists(cur_exp_name + "_" + cur_method + ".json"):
    if row_idx < len(df) :#and is_need_run(save_dir,cur_dataset_name,row_idx,model,num_trial,cur_method):
        print("row_idx:", cur_dataset_name,row_idx,model,cur_method)
        is_done = False
        while not is_done:
            #try:
            prev_outputs = []
            input_nl = df.iloc[row_idx]["NL"]
            print("input:")
            print(input_nl)
            ap_dict, ollama_ap_output = data_loader.load_vars(data_home_dir, cur_dataset_name, row_idx=row_idx, always_generate=ALWAYS_GENERATE_VARIABLES, model=model)
            if ollama_ap_output is not None:
                print("Ollama-generated atomic propositions (used for formalization):")
                print(ollama_ap_output)
            if cur_method == "nl2structnl":
                outputs = llm_prompt.get_formalizations_loop(input_nl, ap_dict, nl2structnl.get_nl2structnl_translation, num_trial=num_trial, model=model,prev_outputs=prev_outputs)
            else:
                raise ValueError(f"Unknown method: {cur_method}")
            #for output in outputs:
            #    print(output['output_structured_natural_language'])
            cur_exp_name = f"{save_dir}/{cur_dataset_name}-{row_idx}_model-{model}_trials-{num_trial}"
            with open(cur_exp_name + "_" + cur_method + ".json", "w") as json_file:
                json.dump(outputs, json_file)
            print(row_idx,cur_method,"done!","num outputs:",len(outputs))
            is_done = True
            #except Exception as e:
            #    print(f"Error during generation: {e}")
            #    time.sleep(30)
        return outputs

In [25]:
row_idx_range = None

tasks = []
for cur_dataset_name in dataset_list:
    for model in model_list:
        for method in cur_methods:
            cur_df_file = data_home_dir + cur_dataset_name + "/PlausibleSpecs.xlsx"
            df = pd.read_excel(cur_df_file, engine='openpyxl')
            for row_idx in range(len(df)):
                if row_idx_range is None or row_idx in row_idx_range:
                    tasks.append((cur_dataset_name,model,row_idx,method))

In [26]:
#run the LLM on each requirement in the benchmarks
for cur_dataset_name,model,row_idx, cur_method in tasks:
    outputs = run_single_task(cur_dataset_name,model,row_idx, cur_method)
    print("output:")
    print(outputs)
    #remove break after testing a single run, to process entire dataset
    #after generating outputs, then run compute_accuracy_metrics.ipynb
    #then do Plot_results.ipynb LATER when you have all versions good


row_idx: FSM-S 0 qwen2.5:32b nl2structnl
input:
The sensor shall change states from NOMINAL to FAULT when limits are exceeded.
error msg: please fix your output list after addressing the following problem in the 0-th item: N_DURATION 0 is not a valid non-zero integer
0 nl2structnl done! num outputs: 1
output:
[{'explanation': "The property specifies that when limits are exceeded (i.e., 'limits' is True), the system's sensor state changes from NOMINAL to FAULT. We need to capture this transition: after 'limits' becomes true, the state should no longer be in NOMINAL and instead be in FAULT.", 'decision1': '_ABSTRACT_VAR1_', 'bool_exp1': '', 'decision2': 'upon bool_exp2, _ABSTRACT_VAR2_', 'bool_exp2': 'limits', 'decision3': 'immediately satisfy bool_exp3', 'bool_exp3': '!state_is_NOMINAL & state_is_FAULT', 'bool_exp4': '', 'N_DURATION': None, 'decision1_substring': '', 'decision2_substring': 'when limits are exceeded.', 'decision3_substring': 'change states from NOMINAL to FAULT', 'output